# MDPs, States, Actions & Rewards Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: a tiny deterministic MDP

A 4×4 GridWorld. Agent starts top-left, terminal at bottom-right, reward of -1 per step, actions `{up, down, left, right}`. See `code/main.py`.

In [ ]:
```python

GRID = 4

TERMINAL = (3, 3)

ACTIONS = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}

def step(state, action):

    if state == TERMINAL:

        return state, 0.0, True

    dr, dc = ACTIONS[action]

    r, c = state

    nr = min(max(r + dr, 0), GRID - 1)

    nc = min(max(c + dc, 0), GRID - 1)

    return (nr, nc), -1.0, (nr, nc) == TERMINAL

In [ ]:
```

Five lines. That is the entire environment. Deterministic transitions, constant step penalty, absorbing terminal state.

### Step 2: roll out a policy

A policy is a function from state to action distribution. The simplest: uniform random.

In [ ]:
```python

def uniform_policy(state):

    return {a: 0.25 for a in ACTIONS}

def rollout(policy, max_steps=200):

    s, total, steps = (0, 0), 0.0, 0

    for _ in range(max_steps):

        a = sample(policy(s))

        s, r, done = step(s, a)

        total += r

        steps += 1

        if done:

            break

    return total, steps

In [ ]:
```

Run the random policy 1000 times. Average return is around -60 to -80 for this 4×4 board. The optimal return is -6 (straight-line path down-right). Closing that gap is everything in Phase 9.

### Step 3: compute `V^π` exactly via the Bellman equation

For small MDPs the Bellman equation is a linear system. Enumerate states, apply the expectation, iterate until the values stop changing.

In [ ]:
```python

def policy_evaluation(policy, gamma=0.99, tol=1e-6):

    V = {s: 0.0 for s in all_states()}

    while True:

        delta = 0.0

        for s in all_states():

            if s == TERMINAL:

                continue

            v = 0.0

            for a, pi_a in policy(s).items():

                s_next, r, _ = step(s, a)

                v += pi_a * (r + gamma * V[s_next])

            delta = max(delta, abs(v - V[s]))

            V[s] = v

        if delta < tol:

            return V

In [ ]:
```

This is iterative policy evaluation. It is the first algorithm in Sutton & Barto and the theoretical foundation of every RL method that follows.

### Step 4: `γ` is a hyperparameter with physical meaning

Effective horizon is roughly `1 / (1 - γ)`. `γ = 0.9` → 10 steps. `γ = 0.99` → 100 steps. `γ = 0.999` → 1000 steps.

Too low and the agent acts myopically. Too high and credit assignment becomes noisy, because many early steps share responsibility for far-future reward. LLM RLHF typically uses `γ = 1` because episodes are short and bounded. Control tasks use `0.95–0.99`. Long-horizon strategy games use `0.999`.

## Exercises

In [ ]:
1. **Easy.** Implement the 4×4 GridWorld and random-policy rollout in `code/main.py`. Run 10,000 episodes. Report mean and std of return. Compare to the optimal return (-6).
2. **Medium.** Run `policy_evaluation` with `γ ∈ {0.5, 0.9, 0.99}` for the uniform-random policy. Print `V` as a 4×4 grid for each. Explain why the state values near the terminal grow faster with larger `γ`.
3. **Hard.** Turn the GridWorld stochastic: each action slips to an adjacent direction with probability `p = 0.1`. Re-evaluate the uniform policy. Does `V[start]` get better or worse? Why?